In [0]:
# ============================================================
# BANKING DATA ENGINEERING PROJECT
# GOLD LAYER - TRANSACTION ANALYSIS
# ============================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    when,
    count,
    sum,
    avg
)


# ============================================================
# SPARK SESSION
# ============================================================

spark = SparkSession.builder.appName(
    "Banking_Gold_Transaction_Analysis"
).getOrCreate()


# ============================================================
# CONFIGURATION
# ============================================================

silver_path = (
    "/Volumes/bank_data/default/bank_data/"
    "silver_bank_transactions"
)

gold_path = (
    "/Volumes/bank_data/default/bank_data/"
    "gold_transaction_analysis"
)


# ============================================================
# READ SILVER
# ============================================================

print("========================================")
print("            READ SILVER")
print("========================================")


silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)


silver_count = silver_df.count()

print(f"Silver Rows: {silver_count}")


# ============================================================
# CREATE TRANSACTION TYPE
# ============================================================

print("\n========================================")
print("      CREATE TRANSACTION TYPE")
print("========================================")


transaction_df = (
    silver_df

    .withColumn(
        "Transaction_Type",
        when(
            col("DEPOSIT_AMT").isNotNull()
            & (col("DEPOSIT_AMT") > 0),
            "Deposit"
        )
        .when(
            col("WITHDRAWAL_AMT").isNotNull()
            & (col("WITHDRAWAL_AMT") > 0),
            "Withdrawal"
        )
    )
)


# ============================================================
# GOLD TRANSFORMATION
# ============================================================

print("\n========================================")
print("        GOLD TRANSFORMATION")
print("========================================")


gold_df = (
    transaction_df

    .filter(
        col("Transaction_Type").isNotNull()
    )

    .withColumn(
        "Transaction_Amount",
        when(
            col("Transaction_Type") == "Deposit",
            col("DEPOSIT_AMT")
        )
        .when(
            col("Transaction_Type") == "Withdrawal",
            col("WITHDRAWAL_AMT")
        )
    )

    .groupBy("Transaction_Type")

    .agg(

        count("*").alias(
            "Transaction_Count"
        ),

        sum(
            col("Transaction_Amount")
        ).alias(
            "Total_Amount"
        ),

        avg(
            col("Transaction_Amount")
        ).alias(
            "Avg_Amount"
        )
    )
)


# ============================================================
# WRITE GOLD
# ============================================================

print("\n========================================")
print("            WRITE GOLD")
print("========================================")


(
    gold_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_path)
)


print(
    "Gold Transaction Analysis written successfully"
)


# ============================================================
# READ GOLD FOR VALIDATION
# ============================================================

gold_check_df = (
    spark.read
    .format("delta")
    .load(gold_path)
)


gold_count = gold_check_df.count()

gold_columns = len(
    gold_check_df.columns
)


# ============================================================
# GOLD VALIDATION
# ============================================================

print("\n========================================")
print("          GOLD VALIDATION")
print("========================================")


print(
    f"Silver Rows: {silver_count}"
)

print(
    f"Gold Transaction Rows: {gold_count}"
)

print(
    f"Gold Columns: {gold_columns}"
)


# ============================================================
# EXPECTED RESULTS
# ============================================================

expected_transaction_types = 2

expected_columns = 4


if gold_count == expected_transaction_types:

    print(
        " TRANSACTION TYPE COUNT VALIDATION PASSED"
    )

else:

    print(
        " TRANSACTION TYPE COUNT VALIDATION FAILED"
    )


if gold_columns == expected_columns:

    print(
        " COLUMN COUNT VALIDATION PASSED"
    )

else:

    print(
        " COLUMN COUNT VALIDATION FAILED"
    )


# ============================================================
# NULL VALIDATION
# ============================================================

print("\n========================================")
print("           NULL VALIDATION")
print("========================================")


null_transaction_type = (
    gold_check_df
    .filter(
        col("Transaction_Type").isNull()
    )
    .count()
)


null_transaction_count = (
    gold_check_df
    .filter(
        col("Transaction_Count").isNull()
    )
    .count()
)


null_total_amount = (
    gold_check_df
    .filter(
        col("Total_Amount").isNull()
    )
    .count()
)


print(
    f"NULL Transaction_Type: "
    f"{null_transaction_type}"
)

print(
    f"NULL Transaction_Count: "
    f"{null_transaction_count}"
)

print(
    f"NULL Total_Amount: "
    f"{null_total_amount}"
)


if null_transaction_type == 0:

    print(
        " Transaction_Type NULL CHECK PASSED"
    )

else:

    print(
        " Transaction_Type NULL CHECK FAILED"
    )


if null_transaction_count == 0:

    print(
        "Transaction_Count NULL CHECK PASSED"
    )

else:

    print(
        " Transaction_Count NULL CHECK FAILED"
    )


if null_total_amount == 0:

    print(
        " Total_Amount NULL CHECK PASSED"
    )

else:

    print(
        " Total_Amount NULL CHECK FAILED"
    )


# ============================================================
# TRANSACTION TYPE VALIDATION
# ============================================================

print("\n========================================")
print("     TRANSACTION TYPE VALIDATION")
print("========================================")


transaction_types = (
    gold_check_df
    .select("Transaction_Type")
    .distinct()
    .collect()
)


transaction_type_names = sorted(
    [
        row["Transaction_Type"]
        for row in transaction_types
    ]
)


print(
    f"Transaction Types: "
    f"{transaction_type_names}"
)


expected_types = [
    "Deposit",
    "Withdrawal"
]


if transaction_type_names == expected_types:

    print(
        " TRANSACTION TYPES VALIDATION PASSED"
    )

else:

    print(
        " TRANSACTION TYPES VALIDATION FAILED"
    )


# ============================================================
# BUSINESS VALIDATION
# ============================================================

print("\n========================================")
print("       BUSINESS VALIDATION")
print("========================================")


invalid_counts = (
    gold_check_df
    .filter(
        col("Transaction_Count") <= 0
    )
    .count()
)


invalid_amounts = (
    gold_check_df
    .filter(
        col("Total_Amount") <= 0
    )
    .count()
)


print(
    f"Transaction types with invalid count: "
    f"{invalid_counts}"
)

print(
    f"Transaction types with invalid amount: "
    f"{invalid_amounts}"
)


if invalid_counts == 0:

    print(
        " TRANSACTION COUNT BUSINESS CHECK PASSED"
    )

else:

    print(
        " TRANSACTION COUNT BUSINESS CHECK FAILED"
    )


if invalid_amounts == 0:

    print(
        " TOTAL AMOUNT BUSINESS CHECK PASSED"
    )

else:

    print(
        " TOTAL AMOUNT BUSINESS CHECK FAILED"
    )


# ============================================================
# SCHEMA
# ============================================================

print("\n========================================")
print("             GOLD SCHEMA")
print("========================================")


gold_check_df.printSchema()


# ============================================================
# SAMPLE DATA
# ============================================================

print("\n========================================")
print("          GOLD SAMPLE DATA")
print("========================================")


display(
    gold_check_df.orderBy(
        col("Transaction_Type")
    )
)


# ============================================================
# FINAL DATA QUALITY CHECK
# ============================================================

if (
    gold_count == expected_transaction_types
    and gold_columns == expected_columns
    and null_transaction_type == 0
    and null_transaction_count == 0
    and null_total_amount == 0
    and transaction_type_names == expected_types
    and invalid_counts == 0
    and invalid_amounts == 0
):

    print("\n========================================")
    print("   GOLD TRANSACTION ANALYSIS COMPLETED")
    print("========================================")

    print(
        "ALL GOLD TRANSACTION CHECKS PASSED"
    )

    print(
        f"Transaction Types: {gold_count}"
    )

    print(
        f"Columns: {gold_columns}"
    )

    print(
        f"Gold Delta Path: {gold_path}"
    )

else:

    raise Exception(
        "GOLD TRANSACTION ANALYSIS "
        "DATA QUALITY CHECK FAILED"
    )

            READ SILVER
Silver Rows: 116162

      CREATE TRANSACTION TYPE

        GOLD TRANSFORMATION

            WRITE GOLD
Gold Transaction Analysis written successfully

          GOLD VALIDATION
Silver Rows: 116162
Gold Transaction Rows: 2
Gold Columns: 4
 TRANSACTION TYPE COUNT VALIDATION PASSED
 COLUMN COUNT VALIDATION PASSED

           NULL VALIDATION
NULL Transaction_Type: 0
NULL Transaction_Count: 0
NULL Total_Amount: 0
 Transaction_Type NULL CHECK PASSED
Transaction_Count NULL CHECK PASSED
 Total_Amount NULL CHECK PASSED

     TRANSACTION TYPE VALIDATION
Transaction Types: ['Deposit', 'Withdrawal']
 TRANSACTION TYPES VALIDATION PASSED

       BUSINESS VALIDATION
Transaction types with invalid count: 0
Transaction types with invalid amount: 0
 TRANSACTION COUNT BUSINESS CHECK PASSED
 TOTAL AMOUNT BUSINESS CHECK PASSED

             GOLD SCHEMA
root
 |-- Transaction_Type: string (nullable = true)
 |-- Transaction_Count: long (nullable = true)
 |-- Total_Amount: double (null

Transaction_Type,Transaction_Count,Total_Amount,Avg_Amount
Deposit,62637,2.383687153235002E11,3805557.6627791915
Withdrawal,53525,2.4020113228475958E11,4487643.760574677



   GOLD TRANSACTION ANALYSIS COMPLETED
ALL GOLD TRANSACTION CHECKS PASSED
Transaction Types: 2
Columns: 4
Gold Delta Path: /Volumes/bank_data/default/bank_data/gold_transaction_analysis
